# 👥 04 - Comparing Groups

Module 06 taught the mechanics of `groupby()`, `.agg()`, and `value_counts()`. This notebook uses them for the most common question in data analysis: **"Is this group different from that one?"**

Comparing groups well takes more than the right method call. You need the right **statistic** for the column type (notebook 01), a fair **denominator** (counts vs. rates), enough **data** in each group, and an eye out for a **second variable** hiding behind the first.

## ✅ Learning Goals
- Summarize a numeric column by group with center **and** spread, plus the group size
- Compare a yes/no outcome across groups as a **rate**, not a count
- Build two-way tables with `pd.crosstab()` (counts and row percentages) and `pivot_table()`
- Compare groups made from a numeric column by binning it with `pd.cut()`
- Spot small groups and confounding variables before drawing conclusions

In [1]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})

## 1. Numeric Column by Group: Center, Spread, and Size
A group mean on its own hides too much. A useful group summary always has **how many**, **where the middle is**, and **how spread out**. Named aggregation (Module 06) keeps the column names readable:

In [2]:
fare_by_class = (
    titanic
    .groupby('class')
    .agg(
        passengers=('Fare', 'size'),
        mean_fare=('Fare', 'mean'),
        median_fare=('Fare', 'median'),
        std_fare=('Fare', 'std'),
        max_fare=('Fare', 'max'),
    )
    .round(2)
)
fare_by_class

,passengers,mean_fare,median_fare,std_fare,max_fare
class,,,,,
1st,216,84.15,60.29,78.38,512.33
2nd,184,20.66,14.25,13.42,73.50
3rd,491,13.68,8.05,11.78,69.55


Read it with notebooks 02 and 03 in mind. 1st class has the biggest gap between mean and median, and by far the biggest std: the outlier tickets from notebook 03 all live in this group. To compare **typical** fares across classes, the median is the fairer choice.

### Quick Version: `.describe()` by Group
`groupby(...)[col].describe()` gives the full `.describe()` table for each group in one line:

In [3]:
titanic.groupby('class')['Age'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
class,,,,,,,,
1st,186.0,38.2,14.8,0.9,27.0,37.0,49.0,80.0
2nd,173.0,29.9,14.0,0.7,23.0,29.0,36.0,70.0
3rd,355.0,25.1,12.5,0.4,18.0,24.0,32.0,74.0


✅ **Your Turn:** Build a named-aggregation table of `Age` by `Sex` with the count of **known** ages (`'count'`), the mean, the median, and the std. Why does `'count'` differ from `'size'` here? (Module 06's `.size()` vs. `.count()`.)

In [4]:
# Your turn here
age_by_sex = titanic.groupby('Sex').agg(
    known_ages=('Age', 'count'),
    mean_age=('Age', 'mean'),
    median_age=('Age', 'median'),
    std_age=('Age', 'std'),
)
print(age_by_sex.round(3))
age_denominators = titanic.groupby('Sex').agg(passengers=('Age', 'size'), known_ages=('Age', 'count'))
age_denominators['missing_ages'] = age_denominators['passengers'] - age_denominators['known_ages']
print(age_denominators)
print('count excludes missing ages: female 261 known of 314 (53 missing); male 453 of 577 (124 missing).')
print('size counts every passenger row. Age mean, median and sample std use only known ages.')


        known_ages  mean_age  median_age  std_age
Sex                                              
female         261    27.916        27.0   14.110
male           453    30.727        29.0   14.678
        passengers  known_ages  missing_ages
Sex                                         
female         314         261            53
male           577         453           124
count excludes missing ages: female 261 known of 314 (53 missing); male 453 of 577 (124 missing).
size counts every passenger row. Age mean, median and sample std use only known ages.


## 2. Yes/No Outcome by Group: Rates, Not Counts
Notebook 01 showed that the mean of a 0/1 column is a proportion. Group by something, take the mean of `Survived`, and you get a **survival rate per group**:

In [5]:
titanic.groupby('Sex')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)

,passengers,survived,survival_rate
Sex,,,
female,314,233,0.742038
male,577,109,0.188908


Look at the `survived` and `survival_rate` columns together. 233 women and 109 men survived. Comparing **counts** would already say "women did better," but only because the counts happen to agree with the rates. Counts depend on how big each group is. Rates divide that out.

Here's a case where counts mislead:

In [6]:
by_class = titanic.groupby('class')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)
by_class

,passengers,survived,survival_rate
class,,,
1st,216,136,0.629630
2nd,184,87,0.472826
3rd,491,119,0.242363


More 3rd-class passengers survived (119) than 2nd-class (87). But 3rd class was much bigger, and its survival **rate** was the lowest. "More survivors" and "better chances" are different claims.

> 🧠 Before comparing groups, ask: **is the denominator the same?** If not, compare rates or percentages.

## 3. Two Categorical Columns: `pd.crosstab()`
`pd.crosstab(rows, columns)` counts every combination of two categorical columns:

In [7]:
pd.crosstab(titanic['class'], titanic['Sex'], margins=True)

Sex,female,male,All
class,,,
1st,94,122,216
2nd,76,108,184
3rd,144,347,491
All,314,577,891


`margins=True` adds an `All` row and column of totals. Counts again have the denominator problem, so use `normalize=` to turn them into percentages:

| `normalize=` | Each … adds up to 100% | Answers |
|---|---|---|
| `'index'` | row | "Within each class, what share is female?" |
| `'columns'` | column | "Of all the women, what share was in each class?" |
| `'all'` | whole table | "What share of everyone is a 1st-class woman?" |

In [8]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='index') * 100).round(1)

Sex,female,male
class,,
1st,43.5,56.5
2nd,41.3,58.7
3rd,29.3,70.7


In [9]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='columns') * 100).round(1)

Sex,female,male
class,,
1st,29.9,21.1
2nd,24.2,18.7
3rd,45.9,60.1


Same data, two different questions, two different tables. Before running a crosstab, say the question out loud. The question decides which way to normalize.

## 4. A Statistic for Every Combination: `pivot_table()`
`crosstab` counts. `pivot_table` computes **any** statistic of a third column, for every combination of two others:

In [10]:
survival_table = titanic.pivot_table(
    values='Survived',
    index='class',
    columns='Sex',
    aggfunc='mean',
)
(survival_table * 100).round(1)

Sex,female,male
class,,
1st,96.8,36.9
2nd,92.1,15.7
3rd,50.0,13.5


This table says much more than either one-way table. Women in 1st and 2nd class almost all survived. Men in 2nd and 3rd class almost all didn't. The class gap is large for women (97% vs. 50%) and the sex gap is large in every class.

In Module 08, this exact table becomes a **heatmap**.

## 5. Groups from a Numeric Column: `pd.cut()`
To compare age groups, first turn `Age` into bands with `pd.cut()` (Module 06), then group by the bands.

In [11]:
titanic['age_group'] = pd.cut(
    titanic['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['child', 'teen', 'young adult', 'adult', 'senior'],
)

titanic.groupby('age_group', observed=True)['Survived'].agg(['size', 'mean']).round(3)

,size,mean
age_group,,
child,69,0.580
teen,70,0.429
young adult,358,0.383
adult,195,0.400
senior,22,0.227


`pd.cut()` returns an **ordered category** (notebook 01), so the groups come back in age order. `observed=True` tells pandas to show only the bands that actually have rows.

⚠️ The sizes add up to 714, not 891. Passengers with no recorded age aren't in any band. A survival rate for "children" says nothing about the 177 passengers whose age is unknown.

## 6. Comparing Fairly: Two Traps
### Trap 1: Tiny Groups
A rate from 5 people is much less trustworthy than a rate from 500. Always keep the group size next to the statistic, and be suspicious of extreme rates from small groups:

In [12]:
family = titanic.groupby('SibSp')['Survived'].agg(['size', 'mean']).round(2)
family

,size,mean
SibSp,,
0,608,0.35
1,209,0.54
2,28,0.46
3,16,0.25
4,18,0.17
5,5,0.00
8,7,0.00


The `5` and `8` rows show a 0% survival rate, but they're tiny groups (each one is essentially a single family). One family's fate isn't a pattern. A common fix is to only report groups above a minimum size, using Module 06's group filtering:

In [13]:
family[family['size'] >= 20]

,size,mean
SibSp,,
0,608,0.35
1,209,0.54
2,28,0.46


### Trap 2: A Hidden Second Variable (Confounding)
Passengers who boarded at Cherbourg (`C`) had the highest survival rate:

In [14]:
titanic.groupby('Embarked')['Survived'].agg(['size', 'mean']).round(3)

,size,mean
Embarked,,
C,168,0.554
Q,77,0.390
S,644,0.337


Did boarding at Cherbourg save lives? Check who boarded there:

In [15]:
(pd.crosstab(titanic['Embarked'], titanic['class'], normalize='index') * 100).round(1)

class,1st,2nd,3rd
Embarked,,,
C,50.6,10.1,39.3
Q,2.6,3.9,93.5
S,19.7,25.5,54.8


About half of the Cherbourg passengers were in 1st class, far more than at the other ports. Class affected survival, and class also differed by port. So the port's survival rate is partly a **class** effect in disguise. A variable like `class` that's tangled up with both the group and the outcome is a **confounder**.

The fix is to compare *within* levels of the suspected confounder:

In [16]:
(titanic.pivot_table(values='Survived', index='class', columns='Embarked', aggfunc='mean') * 100).round(1)

Embarked,C,Q,S
class,,,
1st,69.4,50.0,58.3
2nd,52.9,66.7,46.3
3rd,37.9,37.5,19.0


Within each class the port differences are smaller, though not all gone. Grouping alone can't prove what *caused* a difference. It can only show you where to look. Notebook 05 says the same thing about correlation.

---
### 📝 Summary
- Report **size, center, and spread** for every group, not just the mean.
- Compare yes/no outcomes as **rates** (`groupby(...)['col'].mean()`), not counts.
- `pd.crosstab()` counts combinations. `normalize='index'` / `'columns'` answers different questions.
- `pivot_table()` computes any statistic for every combination of two columns.
- Watch for **small groups** (unreliable rates) and **confounders** (a second variable behind the difference).

## 🏋️ Practice

### Practice 1 — Legendary vs. Not (easy)
Load `pokemon.csv` into `poke`. Compare `Total` for Legendary and non-Legendary Pokémon with a named-aggregation table showing the count, mean, median, and std.

In [17]:
poke = pd.read_csv('pokemon.csv')
# Your code here
legendary_total = poke.groupby('Legendary').agg(
    count=('Total', 'count'),
    mean=('Total', 'mean'),
    median=('Total', 'median'),
    std=('Total', 'std'),
)
print(legendary_total.round(3))
print('Legendary Pokemon have higher typical Total here: median 600 versus 425.')
print('There are 65 Legendary and 735 non-Legendary records; this comparison describes the supplied dataset.')


           count     mean  median      std
Legendary                                 
False        735  417.214   425.0  106.760
True          65  637.385   600.0   60.937
Legendary Pokemon have higher typical Total here: median 600 versus 425.
There are 65 Legendary and 735 non-Legendary records; this comparison describes the supplied dataset.


### Practice 2 — Legendary Rate by Generation (medium)
Find the share of Pokémon that are Legendary in each `Generation`, as a percentage. Then make a `pd.crosstab()` of `Generation` vs. `Legendary` with `margins=True`. Which generation has the highest Legendary **rate**, and is it also the one with the most Legendary Pokémon?

In [18]:
# Your code here
legendary_by_generation = poke.groupby('Generation').agg(
    pokemon=('Legendary', 'size'),
    legendary=('Legendary', 'sum'),
    legendary_rate=('Legendary', 'mean'),
)
legendary_by_generation['legendary_percent'] = legendary_by_generation['legendary_rate'] * 100
print(legendary_by_generation.round(3))
generation_legendary_counts = pd.crosstab(poke['Generation'], poke['Legendary'], margins=True)
print(generation_legendary_counts)
print('Generation 3 leads both: 18 Legendary out of 160 records, or 11.25%.')
print('The count and rate rankings happen to agree at the top; their denominators still differ by generation.')


            pokemon  legendary  legendary_rate  legendary_percent
Generation                                                       
1               166          6           0.036              3.614
2               106          5           0.047              4.717
3               160         18           0.112             11.250
4               121         13           0.107             10.744
5               165         15           0.091              9.091
6                82          8           0.098              9.756
Legendary   False  True  All
Generation                  
1             160     6  166
2             101     5  106
3             142    18  160
4             108    13  121
5             150    15  165
6              74     8   82
All           735    65  800
Generation 3 leads both: 18 Legendary out of 160 records, or 11.25%.
The count and rate rankings happen to agree at the top; their denominators still differ by generation.


### Practice 3 — Fair Type Rankings (harder)
Rank `Type 1` by median `Attack`, but **only for types with at least 30 Pokémon**. Show the size and median for the top 5. Then show which types the minimum-size rule removed, and why including them could have been misleading.

In [19]:
# Your code here
attack_by_type = poke.groupby('Type 1').agg(size=('Attack', 'size'), median_attack=('Attack', 'median'))
eligible_attack = attack_by_type[attack_by_type['size'] >= 30].sort_values(
    ['median_attack', 'size'], ascending=[False, False]
)
top_five_attack = eligible_attack.head(5)
removed_types = attack_by_type[attack_by_type['size'] < 30].sort_index()
print('Top five eligible types:')
print(top_five_attack)
print('Removed by the minimum-size rule:')
print(removed_types)


Top five eligible types:
        size  median_attack
Type 1                     
Dragon    32          113.5
Rock      44           95.0
Dark      31           88.0
Ground    32           85.0
Fire      52           84.5
Removed by the minimum-size rule:
          size  median_attack
Type 1                       
Fairy       17           52.0
Fighting    27          100.0
Flying       4           85.0
Ice         24           67.0
Poison      28           74.0
Steel       27           89.0


*Your explanation here.*

**Answer:** Removed types are Fairy (17), Fighting (27), Flying (4), Ice (24), Poison (28), and Steel (27). Smaller groups provide less stable descriptions of a typical member and can change rank substantially when only a few records change; Flying has just four records. Fighting's median Attack is 100 and Steel's is 89, so including them would change the top five despite failing the stated size rule. The cutoff is a reporting rule, not proof that excluded types are weak or that included rankings generalize beyond these records; this file also contains alternate forms rather than independent random samples.


## ✏️ Your Turn
Make a `pivot_table()` of the **median** `Fare` by `class` (rows) and `Embarked` (columns). In a sentence or two, explain how this table supports the confounding story from section 6.

In [20]:
# Your turn here
median_fare_table = titanic.pivot_table(values='Fare', index='class', columns='Embarked', aggfunc='median')
print(median_fare_table.round(4))
# Keep cell sizes alongside the medians, especially for small Q groups.
fare_cell_counts = pd.crosstab(titanic['class'], titanic['Embarked'])
print('Passengers behind each median:')
print(fare_cell_counts)

# Verify denominators and group calculations independently against raw masks.
import numpy as np
assert titanic.shape[0] == 891 and poke.shape[0] == 800
assert titanic['Age'].isna().sum() == 177
assert titanic[['Survived', 'Sex', 'Pclass', 'Fare']].notna().all().all()
assert poke[['Legendary', 'Generation', 'Total', 'Type 1', 'Attack']].notna().all().all()
for sex in ['female', 'male']:
    values = titanic.loc[titanic['Sex'].eq(sex), 'Age'].dropna().to_numpy()
    row = age_by_sex.loc[sex]
    assert row.known_ages == len(values)
    assert np.isclose(row.mean_age, np.mean(values))
    assert np.isclose(row.median_age, np.median(values))
    assert np.isclose(row.std_age, np.std(values, ddof=1))
for flag in [False, True]:
    values = poke.loc[poke['Legendary'].eq(flag), 'Total'].to_numpy()
    row = legendary_total.loc[flag]
    assert row['count'] == len(values)
    assert np.isclose(row['mean'], np.mean(values))
    assert np.isclose(row['median'], np.median(values))
    assert np.isclose(row['std'], np.std(values, ddof=1))
for generation in sorted(poke['Generation'].unique()):
    selected = poke[poke['Generation'].eq(generation)]
    expected = int(selected['Legendary'].sum())
    row = legendary_by_generation.loc[generation]
    assert row.pokemon == len(selected)
    assert row.legendary == generation_legendary_counts.loc[generation, True] == expected
    assert np.isclose(row.legendary_rate, expected / len(selected))
assert generation_legendary_counts.loc['All', 'All'] == 800
assert generation_legendary_counts.loc['All', True] == 65
assert legendary_by_generation['legendary_rate'].idxmax() == 3
assert legendary_by_generation['legendary'].idxmax() == 3
assert top_five_attack.index.tolist() == ['Dragon', 'Rock', 'Dark', 'Ground', 'Fire']
assert top_five_attack['size'].tolist() == [32, 44, 31, 32, 52]
assert top_five_attack['median_attack'].tolist() == [113.5, 95.0, 88.0, 85.0, 84.5]
for type_name, row in attack_by_type.iterrows():
    values = poke.loc[poke['Type 1'].eq(type_name), 'Attack'].to_numpy()
    assert row['size'] == len(values) and np.isclose(row.median_attack, np.median(values))
assert attack_by_type['size'].sum() == 800
assert (eligible_attack['size'] >= 30).all() and (removed_types['size'] < 30).all()
assert fare_cell_counts.to_numpy().sum() == 889  # Two missing embarkation ports.
for class_name in median_fare_table.index:
    for port in median_fare_table.columns:
        selected = titanic[titanic['class'].eq(class_name) & titanic['Embarked'].eq(port)]
        assert len(selected) == fare_cell_counts.loc[class_name, port]
        assert np.isclose(median_fare_table.loc[class_name, port], np.median(selected['Fare']))

# A weighted average of class-specific survival rates reproduces each port's
# crude rate. Changing class weights changes that average without changing
# the within-class outcomes, illustrating the class-composition issue.
port_class = titanic.groupby(['Embarked', 'class']).agg(
    passengers=('Survived', 'size'), survivors=('Survived', 'sum'), rate=('Survived', 'mean')
)
known_ports = titanic.dropna(subset=['Embarked'])
common_class_weights = known_ports['class'].value_counts(normalize=True).sort_index()
port_comparison = []
for port in ['C', 'Q', 'S']:
    groups = port_class.loc[port].reindex(common_class_weights.index)
    port_rows = titanic[titanic['Embarked'].eq(port)]
    weighted_crude = np.average(groups['rate'], weights=groups['passengers'])
    crude = port_rows['Survived'].sum() / len(port_rows)
    standardized = (groups['rate'] * common_class_weights).sum()
    assert np.isclose(weighted_crude, crude)
    assert groups['passengers'].sum() == len(port_rows)
    port_comparison.append({'port': port, 'passengers': len(port_rows),
                            'crude_rate': crude, 'same_class_weights_rate': standardized})
print('Descriptive comparison with common class weights (not a causal estimate):')
print(pd.DataFrame(port_comparison).set_index('port').round(4))
assert np.isclose(common_class_weights.sum(), 1)
assert titanic['age_group'].cat.ordered
assert titanic['age_group'].notna().sum() == 714
assert titanic.groupby('age_group', observed=True).size().sum() == 714
assert titanic['age_group'].cat.categories.tolist() == ['child', 'teen', 'young adult', 'adult', 'senior']
print('All independent group, denominator, aggregation, and weighted-rate checks passed.')


Embarked        C      Q      S
class                          
1st       78.2667  90.00  52.00
2nd       24.0000  12.35  13.50
3rd        7.8958   7.75   8.05
Passengers behind each median:
Embarked   C   Q    S
class                
1st       85   2  127
2nd       17   3  164
3rd       66  72  353


Descriptive comparison with common class weights (not a causal estimate):
      passengers  crude_rate  same_class_weights_rate
port                                                 
C            168      0.5536                   0.4859
Q             77      0.3896                   0.4655
S            644      0.3370                   0.3410
All independent group, denominator, aggregation, and weighted-rate checks passed.


*Your explanation here.*

**Answer:** Fare medians differ strongly by class: at Cherbourg they are about 78.27, 24.00, and 7.90 for 1st, 2nd, and 3rd class, whereas within 3rd class the port medians are close (7.75-8.05); combined with Cherbourg's much larger first-class share (50.6%), this supports class composition as part of the explanation for its higher overall survival. Fare alone does not establish a causal effect: differences remain within some classes, Queenstown has only two first-class and three second-class passengers, and other confounders can still matter.


## 🔥 Challenge (Optional) — Differences Between Groups
"Women survived more" is vague. Make it precise. From `survival_table` (section 4), compute for each class:
1. the **difference** in survival rates (women − men), in percentage points, and
2. the **ratio** of the rates (women ÷ men).

In which class is the gap biggest by each measure? Why can the two measures disagree?

In [21]:
# Optional extension: compute on unrounded rates, then round only for display.
class_order = ['1st', '2nd', '3rd']
sex_order = ['female', 'male']
challenge_rates = survival_table.reindex(index=class_order, columns=sex_order)
challenge_counts = pd.crosstab(titanic['class'], titanic['Sex']).reindex(index=class_order, columns=sex_order)
survival_gaps = pd.DataFrame({
    'women_n': challenge_counts['female'],
    'men_n': challenge_counts['male'],
    'women_survival_pct': challenge_rates['female'] * 100,
    'men_survival_pct': challenge_rates['male'] * 100,
    'difference_pp': (challenge_rates['female'] - challenge_rates['male']) * 100,
    'women_to_men_rate_ratio': challenge_rates['female'] / challenge_rates['male'].where(challenge_rates['male'] > 0),
})
display(survival_gaps.round(4))
print('Largest absolute rate difference:', survival_gaps['difference_pp'].idxmax())
print('Largest rate ratio:', survival_gaps['women_to_men_rate_ratio'].idxmax())
print('Percentage-point ranking:', survival_gaps['difference_pp'].sort_values(ascending=False).index.tolist())
print('Rate-ratio ranking:', survival_gaps['women_to_men_rate_ratio'].sort_values(ascending=False).index.tolist())

# Independent verification from raw CSV counts, using exact rational rates.
import csv
import math
from fractions import Fraction
from collections import defaultdict
with open('titanic.csv', newline='', encoding='utf-8') as handle:
    original_rows = list(csv.DictReader(handle))
raw_outcomes = defaultdict(list)
for row in original_rows:
    if all(row[column] != '' for column in ['Pclass', 'Sex', 'Survived']):
        raw_outcomes[(int(row['Pclass']), row['Sex'])].append(int(row['Survived']))
for class_number, class_name in enumerate(class_order, start=1):
    women = raw_outcomes[(class_number, 'female')]
    men = raw_outcomes[(class_number, 'male')]
    female_rate = Fraction(sum(women), len(women))
    male_rate = Fraction(sum(men), len(men))
    assert male_rate > 0  # A zero denominator would make a rate ratio undefined.
    result = survival_gaps.loc[class_name]
    assert result['women_n'] == len(women) and result['men_n'] == len(men)
    assert math.isclose(result['women_survival_pct'], float(female_rate * 100))
    assert math.isclose(result['men_survival_pct'], float(male_rate * 100))
    assert math.isclose(result['difference_pp'], float((female_rate - male_rate) * 100))
    assert math.isclose(result['women_to_men_rate_ratio'], float(female_rate / male_rate))
assert challenge_counts.to_numpy().sum() == 891
assert survival_gaps['difference_pp'].idxmax() == '2nd'
assert survival_gaps['women_to_men_rate_ratio'].idxmax() == '2nd'
print('Independent raw-CSV/Fraction checks passed for all six group rates and all differences/ratios.')


,women_n,men_n,women_survival_pct,men_survival_pct,difference_pp,women_to_men_rate_ratio
class,,,,,,
1st,94,122,96.8085,36.8852,59.9233,2.6246
2nd,76,108,92.1053,15.7407,76.3645,5.8514
3rd,144,347,50.0000,13.5447,36.4553,3.6915


Largest absolute rate difference: 2nd
Largest rate ratio: 2nd
Percentage-point ranking: ['2nd', '1st', '3rd']
Rate-ratio ranking: ['2nd', '3rd', '1st']
Independent raw-CSV/Fraction checks passed for all six group rates and all differences/ratios.


**Optional challenge interpretation:** Second class has the largest difference by both measures: female survival was **92.11%** versus male survival **15.74%**, a gap of **76.36 percentage points** and a female-to-male rate ratio of **5.85**. First class has a **59.92-point** difference and ratio **2.62**; third class has a **36.46-point** difference and ratio **3.69**.

The maximum agrees here, but the complete rankings differ: first class exceeds third class in percentage points, while third class exceeds first class in the ratio. A percentage-point difference subtracts the rates on their original probability scale; a ratio divides by the male baseline, so a small male rate can produce a large relative ratio even when the absolute difference is smaller. A ratio of 5.85 means 5.85 times the male rate, not 5.85 percentage points or a 585% increase (the relative increase would be about 485%).

These are observed group comparisons with unequal sample sizes: first class has 94 women and 122 men, second class 76 and 108, and third class 144 and 347. They are not adjusted causal effects of sex or class, and the table does not quantify sampling uncertainty.

---
✅ Next up → [Relationships and Correlation](05-Relationships_and_Correlation.ipynb): comparing two **numeric** columns, and the limits of a single correlation number.